In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Load proton density data
proton_data = pd.read_csv("CELIAS_Proton_Monitor_Hourly.csv") 
proton_data['datetime'] = pd.to_datetime(proton_data['datetime'])
proton_data.set_index('datetime', inplace=True)

# Load earthquake data
event_df = pd.read_csv("declustered_earthquake_dataset_usgs.csv")
event_df['time'] = pd.to_datetime(event_df['time'], format='mixed')

# Check if the datetime column is timezone-aware
if event_df['time'].dt.tz is None:
    # Localize to 'UTC' if not already timezone-aware
    event_df['time'] = event_df['time'].dt.tz_localize('UTC')

# Filter earthquakes by magnitude and depth
event_df['date'] = event_df['time'].dt.date
event_df = event_df[(event_df['mag'] > 5.6) & (event_df['depth'] < 60)]

# Ensure proton_data index is timezone-aware if needed
if proton_data.index.tz is None:
    proton_data.index = proton_data.index.tz_localize('UTC')
    
# Fill NaN values in proton_data with the mean of each column
proton_data.fillna(proton_data.mean(), inplace=True)

# Create a new column in proton_data to indicate if an earthquake occurred within 24 hours
proton_data['earthquake'] = 0
for index, row in proton_data.iterrows():
    if event_df[(event_df['time'] >= index) & (event_df['time'] < index + pd.Timedelta(hours=24))].shape[0] > 0:
        proton_data.at[index, 'earthquake'] = 1
        
# Prepare data for LSTM model
sequence_length = 48  # Example: sequence of the last 48 hours (you can adjust this)
data = []
labels = []

for i in range(len(proton_data) - sequence_length):
    data.append(proton_data.iloc[i:i+sequence_length].drop(columns=['earthquake']).values)
    labels.append(proton_data.iloc[i+sequence_length]['earthquake'])

data = np.array(data)
labels = np.array(labels)

# Scale the data
scaler = MinMaxScaler(feature_range=(0, 1))
data = scaler.fit_transform(data.reshape(-1, data.shape[-1])).reshape(data.shape)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

In [2]:
proton_data.isna().sum()

SPEED         0
Np            0
Vth           0
N/S           0
V_He          0
GSE_X         0
GSE_Y         0
GSE_Z         0
RANGE         0
HGLAT         0
HGLONG        0
CRN(E)        0
earthquake    0
dtype: int64

In [5]:
# Define the LSTM model
model = Sequential()

# First LSTM layer
model.add(LSTM(units=50, return_sequences=True, input_shape=(sequence_length, data.shape[-1])))
model.add(Dropout(0.2))

# Second LSTM layer
model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))

# Third LSTM layer
model.add(LSTM(units=50))
model.add(Dropout(0.2))

# Dense layer to output the prediction
model.add(Dense(units=1, activation='sigmoid'))  # Sigmoid activation for binary classification


optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)

# Compile the model
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

# Save the model if needed
model.save('lstm_earthquake_predictor.h5')

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 48, 50)         │        12,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 48, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 48, 50)         │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 48, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,051 (207.23 KB)

 Trainable params: 53,051 (207.23 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
6017/6017 ━━━━━━━━━━━━━━━━━━━━ 233s 37ms/step - accuracy: 0.7147 - loss: 0.6008 - val_accuracy: 0.7188 - val_loss: 0.5936
Epoch 2/50
6017/6017 ━━━━━━━━━━━━━━━━━━━━ 260s 37ms/step - accuracy: 0.7147 - loss: 0.5985 - val_accuracy: 0.7188 - val_loss: 0.5928
Epoch 3/50
6017/6017 ━━━━━━━━━━━━━━━━━━━━ 226s 37ms/step - accuracy: 0.7167 - loss: 0.5961 - val_accuracy: 0.7188 - val_loss: 0.5923
Epoch 4/50
6017/6017 ━━━━━━━━━━━━━━━━━━━━ 228s 38ms/step - accuracy: 0.7159 - loss: 0.5961 - val_accuracy: 0.7188 - val_loss: 0.5927
Epoch 5/50
6017/6017 ━━━━━━━━━━━━━━━━━━━━ 261s 38ms/step - accuracy: 0.7159 - loss: 0.5949 - val_accuracy: 0.7188 - val_loss: 0.5909
Epoch 6/50
6017/6017 ━━━━━━━━━━━━━━━━━━━━ 230s 38ms/step - accuracy: 0.7170 - loss: 0.5919 - val_accuracy: 0.7187 - val_loss: 0.5893
Epoch 7/50
6017/6017 ━━━━━━━━━━━━━━━━━━━━ 259s 38ms/step - accuracy: 0.7162 - loss: 0.5911 - val_accuracy: 0.7184 - val_loss: 0.5873
Epoch 8/50
6017/6017 ━━━━━━━━━━━━━━━━━━━━ 229s 38ms/step - accuracy: 

1505/1505 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.7971 - loss: 0.4247
Test Loss: 0.42656978964805603
Test Accuracy: 0.7957864999771118


In [3]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {i: class_weights[i] for i in range(len(class_weights))}

In [4]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Define the LSTM model
model = Sequential()

# First LSTM layer with tanh activation
model.add(LSTM(units=64, return_sequences=True, activation='tanh', input_shape=(sequence_length, data.shape[-1])))
model.add(Dropout(0.25))

# Second LSTM layer with tanh activation
model.add(LSTM(units=64, return_sequences=True, activation='tanh'))
# model.add(Dropout(0.2))

# Third LSTM layer with tanh activation
model.add(LSTM(units=32, activation='tanh'))
# model.add(Dropout(0.2))

# Add a dense layer after LSTM layers
model.add(Dense(units=32, activation='relu'))  # Relu activation is often used here

# Final dense layer to output the prediction
model.add(Dense(units=1, activation='sigmoid'))  # Sigmoid activation for binary classification

# Optimizer with learning rate and gradient clipping
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)

# Compile the model
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Early stopping callback to stop training when the validation loss doesn't improve for 5 epochs
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model with early stopping
model.fit(X_train, y_train, epochs=2, batch_size=64, validation_data=(X_test, y_test), class_weight=class_weights_dict, callbacks=[early_stopping])

# Save the model if needed
# model.save('lstm_earthquake_predictor_class_weight.h5')

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

C:\Users\STARLINECOMP\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 48, 64)         │        19,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 48, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66,241 (258.75 KB)

 Trainable params: 66,241 (258.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
3009/3009 ━━━━━━━━━━━━━━━━━━━━ 164s 52ms/step - accuracy: 0.4698 - loss: 0.6948 - val_accuracy: 0.4853 - val_loss: 0.6931
Epoch 2/2
3009/3009 ━━━━━━━━━━━━━━━━━━━━ 196s 50ms/step - accuracy: 0.5252 - loss: 0.6914 - val_accuracy: 0.4637 - val_loss: 0.7000
1505/1505 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.4864 - loss: 0.6934
Test Loss: 0.693103551864624
Test Accuracy: 0.48527976870536804


In [8]:
pip install pydot

Note: you may need to restart the kernel to use updated packages.


In [5]:
import tensorflow as tf

# ...

tf.keras.utils.plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)

AttributeError: module 'pydot' has no attribute 'InvocationException'

In [ ]:
daigaku wa doko desu ka? soko desu